# Longitudinal Analyses of Bone Marrow & Plasma Proteomics

##  Proteomics Analyses

- Proteomic profiling performed on longitudinal samples collected across clinical visits.
- Paired, non-parametric statistical tests were used to compare protein abundance between time points within the same individuals.
- Time point–to–time point comparisons were conducted between study visits to assess longitudinal changes.
- Results are summarized using BH multiple testing correction and visualized using paired plots and differential expression summaries.
- Analyses are conducted for plasma proteomics and bone marrow interstitial fluid.
- Spearman correlations and longitudinal plots are shown to indicate correlations between tissue samples. 

In [70]:
require(data.table)
require(ggplot2)
require(dplyr)
require(table1)
require(tidyr)
require(UpSetR)

source('helper_functions_ndmm.R')
# Load necessary libraries
library(reshape2)
suppressPackageStartupMessages(library(ComplexHeatmap))

In [71]:
bm <- readRDS('../../data/olink/MM_BMIF_Olink_Final.rds')

In [72]:
plasma <- readRDS('../../data/olink/MM_Plasma_Olink_Final.rds')
plasma <- plasma[cohort.cohortGuid=='FH1']

# Helper Functions for Longitudinal Pre-post Analyses

## Note on Analyses 
In these longitudinal analyses, the following short-hand naming convention is used to denote the different treatment timepoints in the multiple myeloma study:

- Pre-Tx = Pre-Treatment (Plasma & BMIF)
- PI2C = Post Induction 2 Cycles (Plasma only)
- EI = End Induction (Plasma & BMIF)
- ASCT60d = Post Transplant 60 Days (Plasma only)
- ASCT90d = Post Transplant 90 Days (BMIF only)
- ASCT1y = Post Transplant 1 year (Plasma & BMIF)
- ASCT2y = Post Transplant 2 year (Plasma & BMIF)

For the longitudinal component, we examine differences between the following pairs of timepoints to study immune response and reconstitution: 

- Pre-Trx vs PI2C
- Pre-Trx vs EI
- PI2C vs EI
- EI vs ASCT60d/90d
- EI vs ASCT1y
- EI vs ASCT2y

## Helper Functions

In [73]:
### write function to test proteins 
test_protein <- function(assay, Plot=F, mat, size=2){
  
  ### subset matrix to 
  ### include only protein of interest
  tmp = mat[olink.assay ==assay]
  
  ### filter to the columns of interest
  dat = tmp[,c('olink.NPX_norm','subject.subjectGuid','sample.visitDetails')]
  
  ### it looks like there are 
  ### duplicate runs 
  ### per subject + visit combo
  ### so the choice is to take mean
  dat =reshape2::dcast(dat, 
                       subject.subjectGuid ~ sample.visitDetails, 
                       fun.aggregate = mean, 
                       value.var='olink.NPX_norm')

  if(ncol(dat) <3){
      res = data.frame(
        Assay = assay,
        Pvalue = NA,
        Log2FC = NA,
        N = NA
      )
    return(res)  
  } else{
      ### remove subjects that have no paired data
      #dat = dat[!is.na(dat[,2]) & !is.na(dat[,3]),]
      dat = drop_na(dat)
      
      ### test for differences 
      ### Using a MWU paired test 
      res = data.frame(
        Assay = assay,
        Pvalue = suppressWarnings(wilcox.test(dat[,3], dat[,2], paired=T)$p.value),
        Log2FC = median(dat[,3] - dat[,2]),
        N = nrow(dat)
      )
      
      ### add (optional) plotting function 
      if(Plot){
        df= reshape2::melt(dat)
        p=ggplot(df,
               aes(x=variable,
                   y=value,))+geom_boxplot()+geom_point()+
          geom_line(aes(x=variable,
                        y=value,
                        group=subject.subjectGuid))+
          ggpubr::stat_compare_means(method='wilcox.test', paired=T, size=size)+
          ggtitle(paste(res[1,]))
        #print(p)
        return(p)
        } else{ 
        return(res)
        }
      }
}

In [74]:
#### summarize contrast 
summarize_contrast <- function(tName ='Transplant', proteins, t1, t2, mat){

  contrast_matrix <-  mat[sample.visitDetails %in% c(t1,t2)]
  
  ### Run differential test 
 results <- lapply(
      proteins,
      function(x) {
        tryCatch(
          test_protein(x, Plot = FALSE, mat = contrast_matrix), silent=T)
          }
    )
  ### Remove proteins 
  ### that failed differential testing
  results = rbindlist(results[sapply(results, class) !='try-error'])
  
  ### transform p-values into q-values 
  results$adjP <- p.adjust(results$Pvalue, method='fdr')
  results$constrast = tName
  results$Time1 = t1
  results$Time2 = t2
  results$Contrast = paste(t1, t2, sep='-')
  
  ### Calculate # DEPs
  nhits = sum(results$adjP < 0.05)
    
  return(results)
}


# Plasma Longitudinal Differentials

In [75]:
str(plasma)
table(plasma$sample.visitDetails)

Classes ‘data.table’ and 'data.frame':	212577 obs. of  24 variables:
 $ specimen.specimenGuid              : chr  "PL01319-01" "PL01321-01" "PL01351-01" "PL01319-01" ...
 $ olink.assay_id                     : chr  "OID20865" "OID20865" "OID20865" "OID20866" ...
 $ olink.uniprot_id                   : chr  "Q9UBB4" "Q9UBB4" "Q9UBB4" "Q9H3R2" ...
 $ olink.assay                        : chr  "ATXN10" "ATXN10" "ATXN10" "MUC13" ...
 $ olink.panel                        : chr  "Neurology" "Neurology" "Neurology" "Neurology" ...
 $ olink.plate_id                     : chr  "20211036_SS210189" "20211036_SS210189" "20211036_SS210189" "20211036_SS210189" ...
 $ sample.sampleKitGuid               : chr  "KT01319" "KT01321" "KT01351" "KT01319" ...
 $ olink.norm_offset                  : num  -0.172 -0.172 -0.172 -1.032 -1.032 ...
 $ olink.NPX_norm                     : num  -1.09 -3.73 -2.19 1.87 1.41 ...
 $ olink.LOD_norm                     : num  -2.712 -2.712 -2.712 -0.594 -0.594 ...
 $ sampl


    PreTx      PI2C        EI   ASCT60d   ASCT90d    ASCT1y  Flu_Y1D0  Flu_Y1D7 
    24994     22065     22065     17634      2944     18873     16192     16161 
Flu_Y1D90    ASCT2y  Flu_Y2D0  Flu_Y2D7 Flu_Y2D90 
    19029      8646     14658     13217     16099 

In [76]:
### Define Contrasts of Interest 
proteins <- unique(plasma$olink.assay)

pl_comparisons <- data.frame(
    t1 = c('PreTx','PreTx', 'PI2C','EI','EI','EI','ASCT60d'),
    t2 = c('PI2C', 'EI', 'EI', 'ASCT60d','ASCT1y','ASCT2y','ASCT1y'))

pl_comparisons

t1,t2
<chr>,<chr>
PreTx,PI2C
PreTx,EI
PI2C,EI
EI,ASCT60d
EI,ASCT1y
EI,ASCT2y
ASCT60d,ASCT1y


## Run longitudinal comparisons

In [77]:
plasma_longitudinal_comparisons <- lapply(1:nrow(pl_comparisons),
       function(x)
           summarize_contrast('Induction', proteins,
                                    t1 = pl_comparisons$t1[x],
                                    t2=  pl_comparisons$t2[x],
                              mat=plasma
                             )
           
       )
plasma_longitudinal_comparisons <- rbindlist(plasma_longitudinal_comparisons)

## Re-set factors for comparisons 
plasma_longitudinal_comparisons$Contrast = factor(plasma_longitudinal_comparisons$Contrast,
                                                  levels =c('PreTx-PI2C','PreTx-EI','PI2C-EI',
                                                      'EI-ASCT60d', 'EI-ASCT1y','ASCT60d-ASCT1y','EI-ASCT2y'))

## Save Plasma Results 

In [78]:
write.csv(plasma_longitudinal_comparisons,file='../../data/olink/output/plasma_longitudinal_pairwise_deps.csv')

# BMIF Longitudinal Differentials

In [79]:
bm_proteins <- unique(bm$olink.assay)
length(bm_proteins)

[1] 2856

In [80]:
### Define Contrasts of Interest 
bm_comparisons <- data.frame(
    t1 = c('PreTx','EI','EI','EI','ASCT90d'),
    t2 = c('EI', 'ASCT90d','ASCT1y','ASCT2y','ASCT1y'))

bm_comparisons

t1,t2
<chr>,<chr>
PreTx,EI
EI,ASCT90d
EI,ASCT1y
EI,ASCT2y
ASCT90d,ASCT1y


In [81]:
bm_longitudinal_comparisons <- lapply(1:nrow(bm_comparisons),
       function(x){
           t1 = bm_comparisons$t1[x]
           t2 = bm_comparisons$t2[x]
           print(c(t1,t2))
           summarize_contrast('Induction', bm_proteins,
                                    t1 = t1,t2=  t2, mat=bm)
           })
bm_longitudinal_comparisons <- rbindlist(bm_longitudinal_comparisons)

## Re-set factors for longitudinal comparisons 
bm_longitudinal_comparisons$Contrast = factor(bm_longitudinal_comparisons$Contrast,
                                                  levels =c('PreTx-EI','EI-ASCT90d',
                                                            'EI-ASCT1y','ASCT90d-ASCT1y',
                                                            'EI-ASCT2y'))

[1] "PreTx" "EI"   
[1] "EI"      "ASCT90d"
[1] "EI"     "ASCT1y"
[1] "EI"     "ASCT2y"
[1] "ASCT90d" "ASCT1y" 


## Save Bone Marrow Results

In [82]:
write.csv(bm_longitudinal_comparisons,
          file='../../data/olink/output/boneMarrow_longitudinal_pairwise_deps.csv')